<a href="https://colab.research.google.com/github/Vigneshvar16/NLP_DSA/blob/main/CO2_AT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Q1: Rule-based morphological processing system for a search engine
# Words: analyzing, analysis, analytical   (runs as-is in Google Colab)

words = ["analyzing", "analysis", "analytical"]

# Different spellings (allomorphs) of the same root -> one normalized root
ALLOMORPHS = {"analyz": "analyze", "analys": "analyze", "analyt": "analyze"}

# (suffix to strip, affix morphemes, type, word-class change, meaning)
RULES = [
    ("ing", ["-ing"], "Inflectional", "VERB -> VERB (participle)",
     "ongoing action (final e of analyze dropped)"),
    ("ical", ["-ic", "-al"], "Derivational", "VERB -> ADJECTIVE",
     "relating to analysis (analyt + ic + al)"),
    ("is", ["-is"], "Derivational", "VERB -> NOUN",
     "the act/result of analyzing (analys + is)"),
]

def analyse(word):
    for suffix, affixes, kind, change, meaning in RULES:
        if word.endswith(suffix) and word[:-len(suffix)] in ALLOMORPHS:
            stem = word[:-len(suffix)]
            return stem, ALLOMORPHS[stem], affixes, kind, change, meaning
    return word, word, [], "None", "-", "base form"

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows, notes, index = [], [], {}
for w in words:
    stem, root, affixes, kind, change, meaning = analyse(w)
    rows.append([w, stem, root, " ".join(affixes) or "-", kind, change, root])
    notes.append(f"  {w:<11}: {meaning}")
    index.setdefault(root, []).append(w)

print("RULE-BASED MORPHOLOGICAL PROCESSING - SEARCH ENGINE INDEXING\n")
print_table(["Word", "Stem", "Root", "Affix(es)", "Type", "Word-class change", "Normalized"], rows)
print("\nMeaning of the affixes:")
print("\n".join(notes))
print("\nUnified index entry:")
for root, variants in index.items():
    print(f"  '{root}'  ->  {variants}")

RULE-BASED MORPHOLOGICAL PROCESSING - SEARCH ENGINE INDEXING

+------------+--------+---------+-----------+--------------+---------------------------+------------+
| Word       | Stem   | Root    | Affix(es) | Type         | Word-class change         | Normalized |
+------------+--------+---------+-----------+--------------+---------------------------+------------+
| analyzing  | analyz | analyze | -ing      | Inflectional | VERB -> VERB (participle) | analyze    |
| analysis   | analys | analyze | -is       | Derivational | VERB -> NOUN              | analyze    |
| analytical | analyt | analyze | -ic -al   | Derivational | VERB -> ADJECTIVE         | analyze    |
+------------+--------+---------+-----------+--------------+---------------------------+------------+

Meaning of the affixes:
  analyzing  : ongoing action (final e of analyze dropped)
  analysis   : the act/result of analyzing (analys + is)
  analytical : relating to analysis (analyt + ic + al)

Unified index entry:
  'ana

In [2]:
# Q2: Morphological parser for sentiment analysis
# Words: disagree, agreement, agreeable   (runs as-is in Google Colab)

words = ["disagree", "agreement", "agreeable"]
LEXICON = {"agree"}                       # known base words

# prefix -> (type, meaning, word-class change)
PREFIXES = {"dis": ("Derivational", "negation / opposite of the base", "VERB -> VERB")}
# suffix -> (type, meaning, word-class change)
SUFFIXES = {
    "ment": ("Derivational", "state or result of agreeing", "VERB -> NOUN"),
    "able": ("Derivational", "capable of / willing to agree (pleasant)", "VERB -> ADJECTIVE"),
}

def parse(word):
    prefix = suffix = ""
    base = word
    for p in PREFIXES:                                  # 1) prefix rule
        if word.startswith(p) and word[len(p):] in LEXICON:
            prefix, base = p, word[len(p):]
    for s in SUFFIXES:                                  # 2) suffix rule
        if base.endswith(s) and base[:-len(s)] in LEXICON:
            suffix, base = s, base[:-len(s)]
    return prefix, base, suffix

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows = []
for w in words:
    prefix, base, suffix = parse(w)
    affix = PREFIXES[prefix] if prefix else SUFFIXES[suffix]
    polarity = "Negative" if prefix else "Positive"
    rows.append([w, prefix + "-" if prefix else "-", base, "-" + suffix if suffix else "-",
                 affix[0], affix[1], polarity, base])

print("MORPHOLOGICAL PARSER - SENTIMENT ANALYSIS\n")
print_table(["Word", "Prefix", "Root", "Suffix", "Category", "Semantic interpretation",
             "Sentiment", "Normalized base"], rows)
print("\nWord-class change:")
for w in words:
    prefix, base, suffix = parse(w)
    ch = PREFIXES[prefix][2] if prefix else SUFFIXES[suffix][2]
    print(f"  {w:<10}: {ch}")
print("\nThe prefix dis- reverses sentiment; -ment and -able keep the positive meaning of 'agree'.")

MORPHOLOGICAL PARSER - SENTIMENT ANALYSIS

+-----------+--------+-------+--------+--------------+------------------------------------------+-----------+-----------------+
| Word      | Prefix | Root  | Suffix | Category     | Semantic interpretation                  | Sentiment | Normalized base |
+-----------+--------+-------+--------+--------------+------------------------------------------+-----------+-----------------+
| disagree  | dis-   | agree | -      | Derivational | negation / opposite of the base          | Negative  | agree           |
| agreement | -      | agree | -ment  | Derivational | state or result of agreeing              | Positive  | agree           |
| agreeable | -      | agree | -able  | Derivational | capable of / willing to agree (pleasant) | Positive  | agree           |
+-----------+--------+-------+--------+--------------+------------------------------------------+-----------+-----------------+

Word-class change:
  disagree  : VERB -> VERB
  agreement : 

In [3]:
# Q3: Morphology-based normalization for topic modeling and clustering
# Words: govern, government, governance   (runs as-is in Google Colab)
from collections import Counter

words = ["govern", "government", "governance"]
ROOT = "govern"

# suffix -> (word-class change, meaning)
SUFFIXES = {
    "ment": ("VERB -> NOUN", "result / body that governs"),
    "ance": ("VERB -> NOUN", "act or state of governing"),
}

def decompose(word):
    """Return root and the list of derivational suffixes (innermost first)."""
    layers = []
    while word != ROOT:
        for s in SUFFIXES:
            if word.endswith(s) and word[:-len(s)].startswith(ROOT):
                layers.insert(0, s)
                word = word[:-len(s)]
                break
        else:
            break
    return word, layers

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows = []
for w in words:
    root, layers = decompose(w)
    level = len(layers)                                  # derivational level
    hierarchy = root + " (VERB)"
    for s in layers:
        hierarchy += f" -> +{s} (NOUN)"
    rows.append([w, root, " ".join("-" + s for s in layers) or "-", level, hierarchy, root, root])

print("MORPHOLOGY-BASED NORMALIZATION - TOPIC MODELING\n")
print_table(["Word", "Root", "Affix(es)", "Level", "Derivational hierarchy",
             "Normalized", "Topic-model token"], rows)

# Effect on word counting in a sample document
doc = "the government must govern well and good governance helps the government"
tokens = doc.split()
normalized = [decompose(t)[0] if t in words else t for t in tokens]
before = {t: c for t, c in Counter(tokens).items() if t in words}
after = {t: c for t, c in Counter(normalized).items() if t == ROOT}
print("\nWord counts BEFORE normalization:", before)
print("Word counts AFTER  normalization:", after)
print("\nAll three variants are merged into the single topic term 'govern' for clustering.")

MORPHOLOGY-BASED NORMALIZATION - TOPIC MODELING

+------------+--------+-----------+-------+-------------------------------+------------+-------------------+
| Word       | Root   | Affix(es) | Level | Derivational hierarchy        | Normalized | Topic-model token |
+------------+--------+-----------+-------+-------------------------------+------------+-------------------+
| govern     | govern | -         | 0     | govern (VERB)                 | govern     | govern            |
| government | govern | -ment     | 1     | govern (VERB) -> +ment (NOUN) | govern     | govern            |
| governance | govern | -ance     | 1     | govern (VERB) -> +ance (NOUN) | govern     | govern            |
+------------+--------+-----------+-------+-------------------------------+------------+-------------------+

Word counts BEFORE normalization: {'government': 2, 'govern': 1, 'governance': 1}
Word counts AFTER  normalization: {'govern': 4}

All three variants are merged into the single topic term

In [4]:
# Q4: Morphological parsing and normalization for document classification
# Words: activate, activation, reactivation   (runs as-is in Google Colab)

words = ["activate", "activation", "reactivation"]

ROOT = "act"
LEXICON = {"act", "active", "activate", "activation"}      # valid intermediate words

# prefix -> (meaning, word-class change)
PREFIXES = {"re": ("again / repeated action", "NOUN -> NOUN")}
# (suffix, letter(s) to restore, word-class change, meaning, resulting class)
SUFFIX_RULES = [
    ("ion", "e", "VERB -> NOUN",      "act/process of activating", "NOUN"),
    ("ate", "e", "ADJECTIVE -> VERB", "to make active",            "VERB"),
    ("ive", "",  "VERB -> ADJECTIVE", "tending to act",            "ADJECTIVE"),
]

def parse(word):
    prefix, base = "", word
    for p in PREFIXES:
        if word.startswith(p) and word[len(p):] in LEXICON:
            prefix, base = p, word[len(p):]
    layers = []                                          # peeled from outside in
    while base != ROOT:
        for suf, restore, change, meaning, cls in SUFFIX_RULES:
            cand = base[:-len(suf)] + restore
            if base.endswith(suf) and cand in LEXICON | {ROOT}:
                layers.append((suf, change, meaning, base, cand))
                base = cand
                break
        else:
            break
    return prefix, base, layers[::-1]                    # innermost layer first

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows, details = [], []
for w in words:
    prefix, root, layers = parse(w)
    chain = [root] + [l[3] for l in layers]              # act -> active -> activate ...
    normalized = "activate"                              # verb form shared by all three words
    suffixes = " ".join("-" + l[0] for l in layers)
    parsed = (prefix + "- + " if prefix else "") + root + "".join(" + -" + l[0] for l in layers)
    sequence = " -> ".join(chain) + (f" -> (re-) {w}" if prefix else "")
    rows.append([w, prefix + "-" if prefix else "-", root, suffixes, sequence, normalized, parsed])
    details.append((w, prefix, layers))

print("MORPHOLOGICAL PARSING - DOCUMENT CLASSIFICATION\n")
print_table(["Word", "Prefix", "Root", "Suffixes", "Derivational sequence",
             "Normalized", "Parsed representation"], rows)

print("\nEffect of each modification (inner to outer):")
for w, prefix, layers in details:
    print(f"  {w}")
    for suf, change, meaning, before, after in layers:
        print(f"     -{suf:<4}: {change:<18} {meaning}")
    if prefix:
        print(f"     {prefix}-  : {PREFIXES[prefix][1]:<18} {PREFIXES[prefix][0]}")

MORPHOLOGICAL PARSING - DOCUMENT CLASSIFICATION

+--------------+--------+------+----------------+---------------------------------------------------------------+------------+--------------------------------+
| Word         | Prefix | Root | Suffixes       | Derivational sequence                                         | Normalized | Parsed representation          |
+--------------+--------+------+----------------+---------------------------------------------------------------+------------+--------------------------------+
| activate     | -      | act  | -ive -ate      | act -> active -> activate                                     | activate   | act + -ive + -ate              |
| activation   | -      | act  | -ive -ate -ion | act -> active -> activate -> activation                       | activate   | act + -ive + -ate + -ion       |
| reactivation | re-    | act  | -ive -ate -ion | act -> active -> activate -> activation -> (re-) reactivation | activate   | re- + act + -ive + -ate 

In [5]:
# Q5: Inflectional morphology-based normalization for search optimization
# Words: create, creates, creating   (runs as-is in Google Colab)

words = ["create", "creates", "creating"]
LEXICON = {"create"}                                     # known base words

# (suffix, grammatical category, tense/aspect, tag)
RULES = [
    ("ing", "Present participle", "progressive aspect (ongoing action)", "VBG"),
    ("s",   "3rd person singular", "simple present, singular subject",   "VBZ"),
]

def analyse(word):
    if word in LEXICON:
        return "-", "Base form", "simple present / infinitive", "VB", word
    for suffix, category, tense, tag in RULES:
        if word.endswith(suffix):
            stem = word[:-len(suffix)]
            for cand in (stem, stem + "e"):              # e-restoration: creat -> create
                if cand in LEXICON:
                    return "-" + suffix, category, tense, tag, cand
    return "-", "Unknown", "-", "?", word

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows = []
for w in words:
    suffix, category, tense, tag, root = analyse(w)
    rows.append([w, suffix, category, root, root, f"{root}+{tag}"])

print("INFLECTIONAL NORMALIZATION - SEARCH OPTIMIZATION\n")
print_table(["Word", "Suffix", "Grammatical category", "Root", "Normalized base",
             "Final representation"], rows)

print("\nTense / aspect:")
for w in words:
    print(f"  {w:<9}: {analyse(w)[2]}")
print("\nSpelling rule: creating = creat + -ing, so the final e of 'create' is dropped.")
print("All forms map to the base 'create', so a query for one form retrieves the others.")

INFLECTIONAL NORMALIZATION - SEARCH OPTIMIZATION

+----------+--------+----------------------+--------+-----------------+----------------------+
| Word     | Suffix | Grammatical category | Root   | Normalized base | Final representation |
+----------+--------+----------------------+--------+-----------------+----------------------+
| create   | -      | Base form            | create | create          | create+VB            |
| creates  | -s     | 3rd person singular  | create | create          | create+VBZ           |
| creating | -ing   | Present participle   | create | create          | create+VBG           |
+----------+--------+----------------------+--------+-----------------+----------------------+

Tense / aspect:
  create   : simple present / infinitive
  creates  : simple present, singular subject
  creating : progressive aspect (ongoing action)

Spelling rule: creating = creat + -ing, so the final e of 'create' is dropped.
All forms map to the base 'create', so a query for o